# Sampling below the threshold: clipping vs. the corrected sampler

A diffusion model blurs data until only noise is left and generates by running the blur backwards. In a **phase-space** diffusion model the data are also moved by a flow $\dot z = Bz$ while they are blurred. Here the blur is prescribed by its forward marginals,
$$Z_t = e^{tB}Z_0 + \sigma(t)\,\varepsilon,\qquad \sigma(t) = \sigma_0 e^{rt},$$
and the reverse process needs the matrix
$$G = \dot C - BC - CB^{\mathsf T} = 2\sigma^2\,(rI - \mathrm{Sym}\,B).$$

$G$ is positive semidefinite exactly when $r \ge \kappa = \lambda_{\max}(\mathrm{Sym}\,B)$, the largest stretching rate of the flow [1]. Below $\kappa$, the standard reverse SDE, whose noise is $\sqrt G$, **does not exist**. Two ways to sample anyway:

| sampler | drift (forward time) | noise | exact |
|---|---|---|---|
| clipped reverse SDE | $Bz - G_+ s$ | $\sqrt{G_+}$ | only for $r \ge \kappa$ |
| corrected sampler | $Bz - \tfrac12(G + G_+)\,s$ | $\sqrt{G_+}$ | for every $r$ |

$G_+$ is $G$ with its negative eigenvalues set to zero, and $s = \nabla\log p_t$ is the score. The clipped sampler differs from the corrected one by an extra drift $\tfrac12 G_- s$, a pull toward high density along the stretching direction [1].

This notebook runs both samplers with the **exact** score (no neural network), on data made of three Gaussian blobs, with the same initial noise and the same random increments, so that the only difference is the sampler. It runs in about a minute. **Runtime → Run all.**

[1] T. Namba, *A stretching-rate threshold for phase-space diffusion models with isotropic marginal noise schedules*, [doi:10.5281/zenodo.23181225](https://doi.org/10.5281/zenodo.23181225) · code: [github.com/neginyan/phase-space-diffusion-threshold](https://github.com/neginyan/phase-space-diffusion-threshold)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.linalg import expm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from base64 import b64encode

omega   = 2.0      # harmonic Hamiltonian flow per coordinate, kappa = |1 - omega^2| / 2 = 1.5
ratio   = 0.5      # r / kappa  (try 0.8, 0.9, or 1.2: above 1 the two samplers coincide)
sd      = 0.15     # spread of each data blob
N, K    = 4000, 8000   # particles, Euler-Maruyama steps
s0, sT  = 0.01, 40.0   # noise level at t = 0 and t = T

## 1. The forward process

Phase space $z = (x_1, v_1, x_2, v_2)$; in each coordinate the flow is $\dot x = v$, $\dot v = -\omega^2 x$. Its symmetric part has largest eigenvalue $\kappa = |1-\omega^2|/2$.

In [ ]:
b = np.array([[0.0, 1.0], [-omega**2, 0.0]])
B = np.kron(np.eye(2), b)
SymB = (B + B.T) / 2
lam, V = np.linalg.eigh(SymB)
kappa = lam.max(); r = ratio * kappa; T = np.log(sT / s0) / r
sigma = lambda t: s0 * np.exp(r * t)
G      = lambda t: 2 * sigma(t)**2 * (V @ np.diag(r - lam) @ V.T)
Gplus  = lambda t: 2 * sigma(t)**2 * (V @ np.diag(np.clip(r - lam, 0, None)) @ V.T)
sqrtGp = lambda t: np.sqrt(2) * sigma(t) * (V @ np.diag(np.sqrt(np.clip(r - lam, 0, None))) @ V.T)
print(f'kappa = {kappa:.3f},  r = {r:.3f},  T = {T:.2f}')
print('eigenvalues of G / (2 sigma^2):', np.round(r - lam, 3), ' -> negative directions:', int((r - lam < 0).sum()))

## 2. Data and the exact score

Positions: three Gaussian blobs; momenta: $\mathcal N(0, 0.25^2)$. A Gaussian mixture stays a Gaussian mixture under the forward process, with components $\mathcal N(M\mu_k,\; M S_k M^{\mathsf T} + \sigma^2 I)$, $M = e^{tB}$, so its score is known exactly at every time.

In [ ]:
centres = np.array([[-1.0, -0.55], [1.0, -0.55], [0.0, 0.9]])
w = np.ones(3) / 3
mu = np.zeros((3, 4)); mu[:, 0::2] = centres
S = np.array([np.diag([sd**2, 0.25**2, sd**2, 0.25**2])] * 3)

def mixture_sample(n, rng):
    k = rng.choice(3, n, p=w)
    return mu[k] + np.einsum('nij,nj->ni', np.linalg.cholesky(S)[k], rng.standard_normal((n, 4)))

cache = {}
def score(z, t):
    key = round(float(t), 12)
    if key not in cache:
        M = expm(B * t); m = mu @ M.T
        C = np.einsum('ij,kjl,ml->kim', M, S, M) + sigma(t)**2 * np.eye(4)
        P = np.linalg.inv(C); cache[key] = (m, P, np.log(w) - 0.5 * np.linalg.slogdet(C)[1])
    m, P, c = cache[key]
    d = z[:, None, :] - m[None]; Pd = np.einsum('kij,nkj->nki', P, d)
    logp = c[None] - 0.5 * (d * Pd).sum(-1); logp -= logp.max(1, keepdims=True)
    resp = np.exp(logp); resp /= resp.sum(1, keepdims=True)
    return -(resp[..., None] * Pd).sum(1)

## 3. Run both samplers from the same noise

Both start from the same draw of a Gaussian with the exact mean and covariance of $p_T$, and use the same Brownian increments.

In [ ]:
data = mixture_sample(20000, np.random.default_rng(1))
MT = expm(B * T)
prior_mean = MT @ data.mean(0); prior_cov = MT @ np.cov(data.T) @ MT.T + sigma(T)**2 * np.eye(4)
z_start = np.random.default_rng(10).multivariate_normal(prior_mean, prior_cov, N)
times = np.linspace(T, 0, K + 1)

def run(mode):
    z = z_start.copy(); rng = np.random.default_rng(20); snaps = []
    for k in range(K):
        t, h = times[k], times[k] - times[k + 1]; s = score(z, t)
        drift = z @ B.T - (s @ Gplus(t).T if mode == 'clipped' else 0.5 * s @ (G(t) + Gplus(t)).T)
        z = z - h * drift + np.sqrt(h) * rng.standard_normal(z.shape) @ sqrtGp(t).T
        if k % (K // 100) == 0 or k == K - 1: snaps.append(z[:, 0::2].copy())
    return z, np.array(snaps)

out = {m: run(m) for m in ['clipped', 'corrected']}
snap_t = np.array([times[k + 1] for k in range(K) if k % (K // 100) == 0 or k == K - 1])

def spread(x):
    lab = np.argmin(((x[:, None, :] - centres[None])**2).sum(-1), 1)
    return float(np.mean([x[lab == k].std(0).mean() for k in range(3)]))
truth = mixture_sample(N, np.random.default_rng(5))
print(f"spread within each blob   true data {spread(truth[:, 0::2]):.3f}   "
      f"corrected {spread(out['corrected'][0][:, 0::2]):.3f}   clipped {spread(out['clipped'][0][:, 0::2]):.3f}")
print(f"momentum spread           true data {truth[:, 1::2].std():.3f}   "
      f"corrected {out['corrected'][0][:, 1::2].std():.3f}   clipped {out['clipped'][0][:, 1::2].std():.3f}")

## 4. Movie

In [ ]:
bg, fg, mu_c, cC, cF = '#0b1020', '#e8ecf4', '#9aa3b8', '#ff8a65', '#5ad1a0'
sig_s = sigma(snap_t); idx = [k for k in range(len(snap_t)) if sig_s[k] <= 3.0]
fig = plt.figure(figsize=(10.8, 10.8), dpi=100, facecolor=bg)
fig.text(0.05, 0.955, 'Sampling below the threshold', color=fg, fontsize=26, weight='bold')
fig.text(0.05, 0.922, 'Clipping the reverse SDE vs. the corrected sampler, both with the exact score', color=mu_c, fontsize=15)
axs = {}
for i, (name, col, key) in enumerate([('clipped reverse SDE', cC, 'clipped'), ('corrected sampler', cF, 'corrected')]):
    ax = fig.add_axes([0.05 + i*0.47, 0.42, 0.43, 0.45]); ax.set_facecolor('#05070f')
    for m in centres:
        for j in (1, 2): ax.add_patch(plt.Circle(m, j*sd, fill=False, ls='--', color='w', alpha=0.5 if j == 1 else 0.25, lw=1))
    sc = ax.scatter([], [], s=2, color=col, alpha=0.55)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-1.9, 2.1); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_color(col); sp.set_linewidth(2)
    ax.set_title(name, color=col, fontsize=16, pad=8); axs[key] = sc
tl = fig.text(0.05, 0.392, '', color=mu_c, fontsize=12.5)
fig.text(0.52, 0.392, 'dashed circles: 1σ and 2σ of the true blobs', color=mu_c, fontsize=12.5)
ab = fig.add_axes([0.20, 0.17, 0.62, 0.17], facecolor=bg)
for s_ in ab.spines.values(): s_.set_color('#3a4360')
ab.tick_params(colors=mu_c, labelsize=12)
bars = ab.barh([2, 1, 0], [0, 0, 0], color=[fg, cF, cC], height=0.62)
ab.set_yticks([2, 1, 0]); ab.set_yticklabels(['true data', 'corrected', 'clipped'], color=fg, fontsize=13)
ab.set_xlim(0, 0.19); ab.set_xlabel('spread within each blob (standard deviation)', color=mu_c, fontsize=12)
txts = [ab.text(0, y, '', va='center', color=fg, fontsize=13) for y in [2, 1, 0]]
fig.text(0.05, 0.105, f'Phase-space diffusion with a Hamiltonian drift (stretching rate κ = {kappa:g}) and noise grown at r = {ratio:g}κ.\n'
         'Below κ the standard reverse SDE does not exist; clipping its negative diffusion adds a pull toward\n'
         'high density and shrinks every blob. The corrected sampler keeps the exact marginals.',
         color=fg, fontsize=13.5, va='top', linespacing=1.5)
fig.text(0.95, 0.012, 'T. Namba · doi:10.5281/zenodo.23181225', color=mu_c, fontsize=10.5, ha='right')
truth_sp = spread(truth[:, 0::2])

def update(k):
    for key in axs: axs[key].set_offsets(out[key][1][k])
    tl.set_text(f'reverse time: noise level σ = {sig_s[k]:.2f}' if sig_s[k] > 0.0105 else 'done: noise level σ = 0.01')
    vals = [truth_sp, spread(out['corrected'][1][k]), spread(out['clipped'][1][k])] if sig_s[k] < 0.12 else [truth_sp, 0, 0]
    for bar, v, tx, y in zip(bars, vals, txts, [2, 1, 0]):
        bar.set_width(v); tx.set_position((v + 0.003, y)); tx.set_text(f'{v:.3f}' if v > 0 else '')
    if vals[2] > 0: txts[2].set_text(f'{vals[2]:.3f}  ({100*(vals[2]/truth_sp - 1):+.0f}%)')
    return []

movie = FuncAnimation(fig, update, frames=[idx[0]]*8 + idx + [idx[-1]]*30, interval=100)
movie.save('sampler_demo.mp4', fps=10, dpi=100, savefig_kwargs={'facecolor': bg},
           extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '20'])
plt.close(fig)
HTML('<video controls loop autoplay muted width="640" src="data:video/mp4;base64,%s"></video>'
     % b64encode(open('sampler_demo.mp4', 'rb').read()).decode())

## What is proved and what is observed

* **Proved** [1]: $G \succeq 0 \iff r \ge \kappa$; below $\kappa$ the clipped sampler has wrong intermediate marginals for every data distribution (Proposition 5), and for Gaussian data its output covariance is strictly too small (Proposition 6); the corrected sampler is exact for every $r$ (Theorem 4).
* **Observed here**: with three Gaussian blobs (a non-Gaussian mixture) and the exact score, the clipped sampler shrinks the spread within each blob by about 13% at $r = \kappa/2$, while the corrected sampler reproduces the data. Set `ratio` to 1.2 and the two samplers coincide, as they must.
* **Not claimed**: better sample quality on real data such as images; the point is where the standard SDE tools stop being exact.

The file `sampler_demo.mp4` appears in the Files panel on the left; right-click → Download.